# Multi-Hop RAG
### Chaining retrievals to answer questions whose answer isn't stated anywhere as a single passage

Corpus: `NIST AI RMF (AI.100-1)` + `OWASP Top 10 for LLMs (2025)` in one index. Neither document ever states the connection between them directly — a single-pass retriever can only find facts *inside* one document, not the link *between* the two.

## Step 1: Build the pipeline

In [1]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu pypdf python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

docs = []
for path, source in [("NIST.AI.100-1.pdf", "NIST"), ("OWASP-Top-10-for-LLMs-v2025.pdf", "OWASP")]:
    pages = PyPDFLoader(path).load()
    for p in pages:
        p.metadata["source"] = source
    docs.extend(pages)

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"Loaded {len(docs)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

C:\Users\shiva\AppData\Local\Temp\ipykernel_26092\3495940767.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Loaded 93 pages -> 281 chunks -> 281 vectors


## Step 2: A question that needs a chained fact
*"Which OWASP LLM Top 10 risk category corresponds most directly to the NIST AI RMF's 'Privacy-Enhanced' characteristic of trustworthy AI?"*

To answer this, the retriever must first learn **what NIST means by "Privacy-Enhanced"** (protecting personal data, identity, and autonomy from unwanted disclosure), then use that meaning — not the original wording — to find the matching **OWASP category**. Neither document mentions the other.

In [3]:
query = "Which OWASP LLM Top 10 risk category corresponds most directly to the NIST AI RMF's 'Privacy-Enhanced' characteristic of trustworthy AI?"

## Step 3: Baseline — single-pass retrieval
Retrieving directly on the original question's wording only activates chunks that share its vocabulary — it never actually looks up what "Privacy-Enhanced" means before searching for a matching OWASP category.

In [4]:
BASELINE_PROMPT = """Answer the question using only the following context.

Context:
{context}

Question: {query}
Answer:"""

baseline_docs = vector_store.similarity_search(query, k=4)
baseline_answer = llm.invoke(BASELINE_PROMPT.format(
    context="\n\n".join(d.page_content for d in baseline_docs), query=query
)).content.strip()

print("Sources:", [(d.metadata["source"], d.metadata["page"]) for d in baseline_docs])
print("\nBaseline answer:\n", baseline_answer)

Sources: [('NIST', 16), ('NIST', 0), ('NIST', 8), ('NIST', 44)]

Baseline answer:
 The OWASP LLM Top 10 risk category that corresponds most directly to the NIST AI RMF's 'Privacy-Enhanced' characteristic of trustworthy AI is "Data Privacy."


## Step 4: Decompose into ordered sub-questions
An LLM call breaks the question into hops, each depending on the previous one's answer.

In [5]:
DECOMPOSE_PROMPT = """Break the following question into 2-3 ordered sub-questions, where each later sub-question
depends on the fact found by the earlier one(s). Return exactly one sub-question per line, no numbering.

Question: {query}"""

def decompose(query):
    text = llm.invoke(DECOMPOSE_PROMPT.format(query=query)).content.strip()
    return [line.strip("-* ").strip() for line in text.split("\n") if line.strip()]

sub_questions = decompose(query)
for i, sq in enumerate(sub_questions, 1):
    print(f"Hop {i}: {sq}")

Hop 1: What are the OWASP LLM Top 10 risk categories?
Hop 2: Which of the OWASP LLM Top 10 risk categories specifically addresses privacy concerns?
Hop 3: How does the identified risk category relate to the NIST AI RMF's 'Privacy-Enhanced' characteristic of trustworthy AI?


## Step 5: Hop loop — retrieve, answer, feed forward
Each hop's retrieved facts are appended to the running context, so hop 2's search query is informed by hop 1's answer.

In [6]:
HOP_PROMPT = """Answer the sub-question using only the following context. Be concise -- one or two sentences.

Context:
{context}

Sub-question: {query}
Answer:"""

def run_hops(sub_questions, k=4):
    trace = []
    accumulated = ""
    for sq in sub_questions:
        search_query = f"{accumulated}\n{sq}" if accumulated else sq
        docs = vector_store.similarity_search(search_query, k=k)
        context = "\n\n".join(d.page_content for d in docs)
        partial = llm.invoke(HOP_PROMPT.format(context=context, query=sq)).content.strip()
        trace.append({
            "sub_question": sq,
            "partial_answer": partial,
            "sources": [(d.metadata["source"], d.metadata["page"]) for d in docs],
        })
        accumulated += f"\nFact: {partial}"
    return trace

trace = run_hops(sub_questions)
for i, hop in enumerate(trace, 1):
    print(f"Hop {i}: {hop['sub_question']}")
    print(f"  -> {hop['partial_answer']}")
    print(f"  sources: {hop['sources']}\n")

Hop 1: What are the OWASP LLM Top 10 risk categories?
  -> The OWASP LLM Top 10 risk categories include Improper Output Handling, Overreliance, and Security Misconfiguration, among others.
  sources: [('OWASP', 22), ('OWASP', 37), ('OWASP', 12), ('OWASP', 4)]

Hop 2: Which of the OWASP LLM Top 10 risk categories specifically addresses privacy concerns?
  -> The OWASP LLM Top 10 risk category that specifically addresses privacy concerns is LLM02:2025 Sensitive Information Disclosure.
  sources: [('OWASP', 22), ('OWASP', 12), ('OWASP', 10), ('OWASP', 37)]

Hop 3: How does the identified risk category relate to the NIST AI RMF's 'Privacy-Enhanced' characteristic of trustworthy AI?
  -> The identified risk category of sensitive information disclosure aligns with the NIST AI RMF's 'Privacy-Enhanced' characteristic by emphasizing the need for AI systems to manage privacy risks effectively, thereby safeguarding sensitive data and ensuring compliance with privacy standards. Addressing these ri

## Step 6: Synthesize the final answer from the hop chain

In [7]:
SYNTHESIZE_PROMPT = """Using the chain of facts below, answer the original question directly.

Facts gathered hop by hop:
{facts}

Original question: {query}
Final answer:"""

facts = "\n".join(f"- {hop['partial_answer']}" for hop in trace)
final_answer = llm.invoke(SYNTHESIZE_PROMPT.format(facts=facts, query=query)).content.strip()

print("Multi-hop answer:\n", final_answer)
print("\n--- vs baseline ---\n", baseline_answer)

Multi-hop answer:
 The OWASP LLM Top 10 risk category that corresponds most directly to the NIST AI RMF's 'Privacy-Enhanced' characteristic of trustworthy AI is LLM02:2025 Sensitive Information Disclosure.

--- vs baseline ---
 The OWASP LLM Top 10 risk category that corresponds most directly to the NIST AI RMF's 'Privacy-Enhanced' characteristic of trustworthy AI is "Data Privacy."


## Try it yourself
1. Write your own 2-hop question that chains a NIST concept to an OWASP category and check whether `decompose` splits it the way you expect.
2. Force a 3-hop chain and watch latency grow — each hop is one retrieval + one LLM call.
3. Corrupt hop 1's answer on purpose (edit `trace[0]["partial_answer"]`) and re-run synthesis to see how a wrong early hop derails the final answer — the error-compounding limitation from the slides.